In [3]:
from dash import Dash, dash_table, html
from dash import Input, Output, ctx
import pandas as pd
import json

app = Dash(__name__)

# ------------------------------------------------------------------
# Sample data
# Each record has a unique ID.
# ------------------------------------------------------------------
df = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106, 107, 108],
    "Name": ["Ali", "Sara", "Reza", "Maryam",
             "Tom", "Zhina", "Boshra", "Jack"],
    "Age": [20, 31, 28, 24, 26, 33, 21, 29],
    "Score": [80, 95, 76, 88, 92, 70, 85, 99]
})

app.layout = html.Div([

    dash_table.DataTable(

        id="tbl",

        data=df.to_dict("records"),

        # Define the columns to display in the DataTable.
        # - "id"   : The key in each row of the data.
        # - "name" : The column header shown to the user.
        # This also controls the order and visibility of the columns.
        columns=[
            {"name": "ID", "id": "id"},
            {"name": "Name", "id": "Name"},
            {"name": "Age", "id": "Age"},
            {"name": "Score", "id": "Score"},
        ],

        # Allow users to edit cell values directly.
        editable=True,

        # Enable row and column selection.
        row_selectable="multi",
        column_selectable="multi",
        
        # Enable built-in sorting and filtering.
        sort_action="native",
        filter_action="native",
        
        # Enable built-in pagination.
        page_action="native",
        page_size=4,

        # Export
        export_format="csv",
        export_headers="display",
    ),

    html.Hr(),

    html.H3("DataTable Inspector"),

    # Display all DataTable properties in a formatted JSON view.
    html.Pre(id="debug")

])


@app.callback(

    Output("debug", "children"),

    Input("tbl", "data"),
    Input("tbl", "data_previous"),
    Input("tbl", "columns"),

    Input("tbl", "selected_rows"),
    Input("tbl", "selected_row_ids"),

    Input("tbl", "selected_columns"),

    Input("tbl", "active_cell"),

    Input("tbl", "selected_cells"),

    Input("tbl", "start_cell"),
    Input("tbl", "end_cell"),

    Input("tbl", "derived_virtual_data"),
    Input("tbl", "derived_virtual_indices"),
    Input("tbl", "derived_virtual_row_ids"),

    Input("tbl", "derived_virtual_selected_rows"),
    Input("tbl", "derived_virtual_selected_row_ids"),

    Input("tbl", "derived_viewport_data"),
    Input("tbl", "derived_viewport_indices"),
    Input("tbl", "derived_viewport_row_ids"),

    Input("tbl", "derived_viewport_selected_rows"),
    Input("tbl", "derived_viewport_selected_row_ids"),

)
def inspector(

    data,
    data_previous,
    columns,

    selected_rows,
    selected_row_ids,

    selected_columns,

    active_cell,

    selected_cells,

    start_cell,
    end_cell,

    derived_virtual_data,
    derived_virtual_indices,
    derived_virtual_row_ids,

    derived_virtual_selected_rows,
    derived_virtual_selected_row_ids,

    derived_viewport_data,
    derived_viewport_indices,
    derived_viewport_row_ids,

    derived_viewport_selected_rows,
    derived_viewport_selected_row_ids,
):

    # ----------------------------------------------------------
    # Which property triggered the callback?
    # ----------------------------------------------------------
    trigger = ctx.triggered_id

    trigger_property = None

    if ctx.triggered:
        trigger_property = ctx.triggered[0]["prop_id"]

    # ----------------------------------------------------------
    # Information about the active row
    # ----------------------------------------------------------
    clicked_row_index = None
    clicked_row_id = None
    clicked_row_data = None

    if active_cell is not None:

        clicked_row_index = active_cell["row"]

        clicked_row_id = active_cell.get("row_id")

        # Find the corresponding record using its ID.
        for row in data:
            if row["id"] == clicked_row_id:
                clicked_row_data = row
                break

    result = {

        # Callback information
        "trigger_component": trigger,
        "trigger_property": trigger_property,

        # Clicked row
        "clicked_row_index": clicked_row_index,
        "clicked_row_id": clicked_row_id,
        "clicked_row_data": clicked_row_data,

        # Active cell
        "active_cell": active_cell,

        # Selected cells
        "selected_cells": selected_cells,
        "start_cell": start_cell,
        "end_cell": end_cell,

        # Selected rows
        "selected_rows": selected_rows,
        "selected_row_ids": selected_row_ids,

        # Selected columns
        "selected_columns": selected_columns,

        # Table structure
        "columns": columns,

        # Raw data
        "data": data,
        "data_previous": data_previous,

        # After filtering/sorting
        "derived_virtual_data": derived_virtual_data,
        "derived_virtual_indices": derived_virtual_indices,
        "derived_virtual_row_ids": derived_virtual_row_ids,

        "derived_virtual_selected_rows":
            derived_virtual_selected_rows,

        "derived_virtual_selected_row_ids":
            derived_virtual_selected_row_ids,

        # Current page
        "derived_viewport_data": derived_viewport_data,
        "derived_viewport_indices": derived_viewport_indices,
        "derived_viewport_row_ids": derived_viewport_row_ids,

        "derived_viewport_selected_rows":
            derived_viewport_selected_rows,

        "derived_viewport_selected_row_ids":
            derived_viewport_selected_row_ids,
    }

    return json.dumps(result, indent=4)


if __name__ == '__main__':
    app.run(debug=True, port = "9871")